In [174]:
import numpy as np
from scipy import sparse

import gurobipy as gp
from gurobipy import GRB
from scipy.sparse.linalg import lsmr, lsqr
import time

In [175]:
def generate_quantile_instance(
    m: int,
    p: int,
    density: float = 0.8,
    seed: int | None = None,
):
    """
    Generate one synthetic Quantile Regression instance.

    Parameters
    ----------
    m : int
        Number of observations / records.

    p : int
        Number of explanatory variables (columns of D).

    density : float, default=0.8
        Target proportion of nonzero entries in D.

    seed : int or None, default=None
        Random seed for reproducibility.

    Returns
    -------
    D : ndarray, shape (m, p)
        Design matrix.

    b : ndarray, shape (m,)
        Dependent variable.
    """

    # create a local random number generator
    rng = np.random.default_rng(seed)

    # check input values
    if m <= 0:
        raise ValueError("m must be positive")

    if p <= 0:
        raise ValueError("p must be positive")

    if not 0.0 <= density <= 1.0:
        raise ValueError("density must lie between 0 and 1")

    # initialize D as an all-zero matrix
    D = np.zeros((m, p), dtype=float)

    # calculate the total number of entries
    n_total = m * p

    # calculate the number of nonzero entries
    n_nonzero = int(round(density * n_total))

    # randomly choose the positions of nonzero entries
    nonzero_indices = rng.choice(n_total, size=n_nonzero, replace=False)

    # fill the selected positions with values sampled from U[-1, 1]
    D.flat[nonzero_indices] = rng.uniform(-1.0, 1.0, size=n_nonzero)

    # generate the dependent variable from U[-1, 1]
    b = rng.uniform(-1.0, 1.0, size=m)

    return D, b

In [176]:
# generate one quantile regression instance
D, b = generate_quantile_instance(m=5000, p=399, density=0.8, seed=42)

# check dimensions
print("D shape:", D.shape)
print("b shape:", b.shape)

# check density
actual_density = np.count_nonzero(D) / D.size

print("number of entries in D:", D.size)
print("number of nonzero entries:", np.count_nonzero(D))
print("actual density:", actual_density)

# check value ranges
print("D min:", D.min())
print("D max:", D.max())
print("b min:", b.min())
print("b max:", b.max())

D shape: (5000, 399)
b shape: (5000,)
number of entries in D: 1995000
number of nonzero entries: 1596000
actual density: 0.8
D min: -0.9999928141121515
D max: 0.9999998390380078
b min: -0.9994594203105618
b max: 0.9998598619646724


In [177]:
def build_quantile_lp(
    D: np.ndarray,
    b: np.ndarray,
    tau: float = 0.2,
):
    """
    Build the LP formulation of a quantile regression instance.

    Parameters
    ----------
    D : ndarray, shape (m, p)
        Design matrix

    b : ndarray, shape (m,)
        Dependent variable

    tau : float, default=0.2
        Quantile level

    Returns
    -------
    A : sparse matrix, shape (m, p + 2m)
        Equality constraint matrix

    c : ndarray, shape (p + 2m,)
        Objective coefficient vector

    bounds : list
        Variable bounds for beta, u_plus, and u_minus
    """

    # get the dimensions of D
    m, p = D.shape

    # check the dimension of b
    if b.shape != (m,):
        raise ValueError("b must have shape (m,)")

    # check the quantile level
    if not 0.0 < tau < 1.0:
        raise ValueError("tau must lie strictly between 0 and 1")

    # convert D to sparse format
    D_sparse = sparse.csr_matrix(D)

    # construct a sparse identity matrix
    I = sparse.eye(m, format="csr")

    # construct A = [D | I | -I]
    A = sparse.hstack((D_sparse, I, -I), format="csr")

    # construct the objective coefficient vector
    c = np.concatenate(
        (np.zeros(p), tau * np.ones(m), (1.0 - tau) * np.ones(m))
    )

    return A, c

In [178]:
# build the quantile regression LP
A, c = build_quantile_lp(D, b, tau=0.2)

# check dimensions
print("A shape:", A.shape)
print("c shape:", c.shape)

# check the three blocks of the objective
print("beta coefficients:", np.unique(c[: D.shape[1]]))
print(
    "u_plus coefficients:", np.unique(c[D.shape[1] : D.shape[1] + D.shape[0]])
)
print("u_minus coefficients:", np.unique(c[D.shape[1] + D.shape[0] :]))

A shape: (5000, 10399)
c shape: (10399,)
beta coefficients: [0.]
u_plus coefficients: [0.2]
u_minus coefficients: [0.8]


In [179]:
def generate_projection_matrix(
    A,
    k: int,
    seed: int | None = None,
):
    """
    Generate a sparse random projection matrix T.

    Parameters
    ----------
    A : sparse matrix, shape (m, n)
        Constraint matrix

    k : int
        Projected dimension

    seed : int or None, default=None
        Random seed for reproducibility

    Returns
    -------
    T : sparse matrix, shape (k, m)
        Sparse random projection matrix
    """

    # create a local random number generator
    rng = np.random.default_rng(seed)

    # get the number of original constraints
    m = A.shape[0]

    # check the projected dimension
    if not 0 < k < m:
        raise ValueError("k must lie strictly between 0 and m")

    # calculate the density of A
    density_A = A.nnz / (A.shape[0] * A.shape[1])

    # set the density of T
    sigma = density_A / 2.0

    # calculate the total number of entries in T
    n_total = k * m

    # calculate the number of nonzero entries in T
    n_nonzero = int(round(sigma * n_total))

    # randomly choose the positions of nonzero entries
    nonzero_indices = rng.choice(n_total, size=n_nonzero, replace=False)

    # convert the flat indices to row and column indices
    rows = nonzero_indices // m
    cols = nonzero_indices % m

    # calculate the standard deviation of nonzero entries
    std = 1.0 / np.sqrt(k * sigma)

    # sample the nonzero entries from the Gaussian distribution
    values = rng.normal(loc=0.0, scale=std, size=n_nonzero)

    # construct the sparse projection matrix
    T = sparse.csr_matrix((values, (rows, cols)), shape=(k, m))

    return T

In [180]:
# choose the projection error parameter
epsilon = 0.2

# get the number of original constraints
m = A.shape[0]

# get the total number of decision variables
n_variables = A.shape[1]

# calculate the projected dimension used in the paper
k = int(round((1.0 / epsilon) ** 2) * np.log(float(n_variables)))

# check the projected dimension
if not 0 < k < m:
    raise ValueError(
        f"k must lie strictly between 0 and m; " f"received k={k} and m={m}"
    )

# generate a sparse random projection matrix
start_time = time.perf_counter()

T = generate_projection_matrix(A, k=k, seed=42)

time_T_generation = time.perf_counter() - start_time

# check the experiment settings
print("epsilon:", epsilon)
print("number of original constraints m:", m)
print("number of decision variables:", n_variables)
print("projected dimension k:", k)
print("projection ratio k/m:", k / m)

# check dimensions
print("T shape:", T.shape)

# check the number of nonzero entries
print("T nnz:", T.nnz)

# check the density of A
density_A = A.nnz / (A.shape[0] * A.shape[1])

print("A density:", density_A)

# check the target density of T
sigma = density_A / 2.0

print("target T density:", sigma)

# check the actual density of T
density_T = T.nnz / (T.shape[0] * T.shape[1])

print("actual T density:", density_T)

# check the projection matrix generation time
print("T generation time:", time_T_generation)

epsilon: 0.2
number of original constraints m: 5000
number of decision variables: 10399
projected dimension k: 231
projection ratio k/m: 0.0462
T shape: (231, 5000)
T nnz: 17838
A density: 0.030887585344744686
target T density: 0.015443792672372343
actual T density: 0.015444155844155844
T generation time: 0.0017699580639600754


In [181]:
def project_lp(
    A,
    b: np.ndarray,
    T,
):
    """
    Project the equality constraints of an LP.

    Parameters
    ----------
    A : sparse matrix, shape (m, n)
        Original constraint matrix

    b : ndarray, shape (m,)
        Original right-hand side vector

    T : sparse matrix, shape (k, m)
        Random projection matrix

    Returns
    -------
    A_projected : sparse matrix, shape (k, n)
        Projected constraint matrix

    b_projected : ndarray, shape (k,)
        Projected right-hand side vector
    """

    # check the dimensions of A and b
    if b.shape != (A.shape[0],):
        raise ValueError("b must have shape (m,)")

    # check the dimensions of T and A
    if T.shape[1] != A.shape[0]:
        raise ValueError("T and A have incompatible dimensions")

    # project the constraint matrix
    A_projected = T @ A

    # project the right-hand side vector
    b_projected = T @ b

    return A_projected, b_projected

In [182]:
def solve_lp(
    A,
    b: np.ndarray,
    c: np.ndarray,
    lb: np.ndarray,
    ub: np.ndarray,
    method: int = -1,
    model_name: str = "lp_model",
):
    """
    Solve a linear program using Gurobi.

    Parameters
    ----------
    A : sparse matrix, shape (m, n)
        Equality constraint matrix

    b : ndarray, shape (m,)
        Right-hand side vector

    c : ndarray, shape (n,)
        Objective coefficient vector

    lb : ndarray, shape (n,)
        Lower bounds of the decision variables

    ub : ndarray, shape (n,)
        Upper bounds of the decision variables

    method : int, default=-1
        Gurobi solution method:
        -1 = automatic
         0 = primal simplex
         1 = dual simplex
         2 = barrier

    model_name : str, default="lp_model"
        Name used for the Gurobi model

    Returns
    -------
    x_opt : ndarray, shape (n,)
        Optimal solution

    objective_value : float
        Optimal objective value
    """

    # get the number of constraints and decision variables
    m, n = A.shape

    # check the dimension of b
    if b.shape != (m,):
        raise ValueError(f"b must have shape {(m,)}")

    # check the dimension of c
    if c.shape != (n,):
        raise ValueError(f"c must have shape {(n,)}")

    # check the dimension of the lower bounds
    if lb.shape != (n,):
        raise ValueError(f"lb must have shape {(n,)}")

    # check the dimension of the upper bounds
    if ub.shape != (n,):
        raise ValueError(f"ub must have shape {(n,)}")

    # check the Gurobi solution method
    if method not in (-1, 0, 1, 2):
        raise ValueError("method must be -1, 0, 1, or 2")

    # create a Gurobi model
    model = gp.Model(model_name)

    # set the Gurobi solution method
    model.Params.Method = method

    # display the optimization log
    model.Params.OutputFlag = 1

    # create the decision variables
    x = model.addMVar(shape=n, lb=lb, ub=ub, name="x")

    # set the objective function
    model.setObjective(c @ x, GRB.MINIMIZE)

    # add the equality constraints
    model.addConstr(A @ x == b, name="equality_constraints")

    # solve the linear program
    model.optimize()

    # check the optimization status
    if model.Status != GRB.OPTIMAL:
        raise RuntimeError(
            "Gurobi did not find an optimal solution; "
            f"status={model.Status}"
        )

    # extract the optimal solution
    x_opt = x.X.copy()

    # extract the optimal objective value
    objective_value = model.ObjVal

    return x_opt, objective_value

In [183]:
def build_quantile_bounds(
    m: int,
    p: int,
):
    """
    Build the variable bounds for the quantile regression LP.

    Parameters
    ----------
    m : int
        Number of observations

    p : int
        Number of regression coefficients

    Returns
    -------
    lb : ndarray, shape (p + 2m,)
        Lower bounds

    ub : ndarray, shape (p + 2m,)
        Upper bounds
    """

    # construct the lower bounds
    lb = np.concatenate((np.full(p, -np.inf), np.zeros(m), np.zeros(m)))

    # construct the upper bounds
    ub = np.full(p + 2 * m, GRB.INFINITY)

    return lb, ub

In [184]:
# project the original equality constraints
start_time = time.perf_counter()
A_projected, b_projected = project_lp(A, b, T)
time_projection = time.perf_counter() - start_time

# get the dimensions of the quantile regression instance
m, p = D.shape

# build the variable bounds
lb, ub = build_quantile_bounds(m=m, p=p)

# solve the projected LP using Gurobi dual simplex
start_time = time.perf_counter()
x_tilde, obj_tilde = solve_lp(A_projected, b_projected, c, lb, ub, method=1)
time_TLP_solve = time.perf_counter() - start_time

Set parameter Method to value 1
Set parameter OutputFlag to value 1
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
Method  1

Optimize a model with 231 rows, 10399 columns and 127845 nonzeros (Min)
Model fingerprint: 0xb0c8c1be
Model has 10000 linear objective coefficients
Coefficient statistics:
  Matrix range     [2e-05, 1e+01]
  Objective range  [2e-01, 8e-01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e-02, 8e+00]

Presolve removed 0 rows and 5325 columns
Presolve time: 0.05s
Presolved: 231 rows, 5074 columns, 112635 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0      handle free variables                          0s
     231    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 231 iterations and 0.07 seconds (0.18 work units)
Optimal objective  0.000000000e+00


In [185]:
def solve_projection_system(
    x0: np.ndarray,
    A,
    b: np.ndarray,
    x_check: np.ndarray,
):
    """
    Approximate the projection of x0 onto A x = b using LSMR.
    """

    # solve A x_hat = b + A x0
    x_hat = lsmr(A, b + A @ x0)[0]

    # construct a second point approximately satisfying A x = b
    x_prime = x_hat - x0

    # construct the direction from x_prime to x_check
    direction = x_check - x_prime

    # handle the degenerate case
    if direction @ direction == 0.0:
        return x_prime

    # calculate the projection coefficient
    lambda_value = (direction @ (x0 - x_prime)) / (direction @ direction)

    # project x0 onto the line through x_prime and x_check
    x_dagger = x_prime + lambda_value * direction

    return x_dagger

In [ ]:
def bound_error_sum(
    x: np.ndarray,
    lb: np.ndarray,
    ub: np.ndarray,
):
    """
    Calculate the sum of bound violations used in the authors' public code.
    """

    # calculate lower-bound violations
    lower_error = np.maximum(lb - x, 0.0)

    # calculate upper-bound violations
    upper_error = np.maximum(x - ub, 0.0)

    # return the sum of all bound violations
    return float(np.sum(lower_error) + np.sum(upper_error))


def retrieval_feasibility(
    x,
    A,
    b,
    lb,
    ub,
    tol=0.01,
):
    """
    Check feasibility using the error measure and tolerance
    used in the authors' public retrieval code.
    """

    # calculate the L2 equality residual
    equality_error = float(np.linalg.norm(A @ x - b))

    # calculate the sum of all bound violations
    bound_error = bound_error_sum(
        x,
        lb,
        ub,
    )

    # apply the authors' combined error criterion
    return bound_error + equality_error <= 2.0 * tol


def retrieval_start(
    x_projected: np.ndarray,
    A,
    b: np.ndarray,
):
    """
    Apply the LSMR retrieval step used before the authors' APM code.
    """

    # calculate the original equality residual of the projected LP solution
    residual = b - A @ x_projected

    # approximately solve A x_part = b - A x_projected
    x_part = lsmr(A, residual)[0]

    # construct the starting point used by the authors' APM code
    x_start = x_projected + x_part

    return x_start


def retrieve_apm(
    x0: np.ndarray,
    A,
    b: np.ndarray,
    lb: np.ndarray,
    ub: np.ndarray,
    max_iter: int = 30,
    delta: float = 0.01,
):
    """
    Apply the von Neumann APM logic from the authors' public code.
    """

    # first apply the authors' LSMR retrieval step
    x = retrieval_start(x0, A, b)

    # use the authors' default status until another stopping condition is met
    status = "iteration limit"

    for iteration in range(1, max_iter + 1):
        # store the point at the beginning of the iteration
        x_previous = x.copy()

        # calculate the bound-error sum before the bound projection
        bound_error = bound_error_sum(x, lb, ub)

        # the authors stop immediately if there is no material bound error
        if bound_error <= delta:
            status = "no bound error"
            return x_previous, iteration, status

        # project onto the bound set
        x = np.clip(x, lb, ub)

        # calculate the equality residual after the bound projection
        equality_error = float(np.linalg.norm(A @ x - b))

        # project onto A x = b when the equality residual is too large
        if equality_error > delta:
            rhs = b + A @ x
            x = lsmr(A, rhs)[0]

        # calculate the authors' no-progress measure
        progress_solution = float(np.linalg.norm(x - x_previous))

        # stop when there is no sufficient progress
        if progress_solution <= delta:
            status = "no sufficient progress"
            return x, iteration, status

        # apply the authors' combined error stopping rule
        if bound_error + equality_error <= 2.0 * delta:
            status = "small total error"
            return x, iteration, status

    return x, max_iter, status

In [187]:
def evaluate_retrieval(
    x: np.ndarray,
    A,
    b: np.ndarray,
    c: np.ndarray,
    lb: np.ndarray,
    ub: np.ndarray,
):
    """
    Evaluate a retrieved solution.
    """

    # calculate the objective value
    objective_value = c @ x

    # calculate the equality constraint residual
    equality_residual = np.linalg.norm(A @ x - b)

    # calculate lower-bound violations
    lower_violation = np.maximum(lb - x, 0.0)

    # calculate upper-bound violations
    upper_violation = np.zeros_like(x)

    finite_upper = np.isfinite(ub)

    upper_violation[finite_upper] = np.maximum(
        x[finite_upper] - ub[finite_upper], 0.0
    )

    # calculate the maximum bound violation
    bound_violation = max(np.max(lower_violation), np.max(upper_violation))

    return (objective_value, equality_residual, bound_violation)

In [188]:
def evaluate_errors(
    x: np.ndarray,
    A,
    b: np.ndarray,
    m: int,
    p: int,
):
    """
    Calculate the average feasibility errors reported in Table 5.

    Parameters
    ----------
    x : ndarray, shape (p + 2m,)
        Retrieved quantile regression solution

    A : sparse matrix, shape (m, p + 2m)
        Original equality constraint matrix

    b : ndarray, shape (m,)
        Original right-hand side vector

    m : int
        Number of observations

    p : int
        Number of regression coefficients

    Returns
    -------
    average_inequality_error : float
        Average nonnegativity violation of u_plus and u_minus

    average_equality_error : float
        Average absolute violation of A x = b
    """

    # check the dimension of the retrieved solution
    if x.shape != (p + 2 * m,):
        raise ValueError(f"x must have shape {(p + 2 * m,)}")

    # check the dimension of b
    if b.shape != (m,):
        raise ValueError(f"b must have shape {(m,)}")

    # extract the positive residual variables
    u_plus = x[p : p + m]

    # extract the negative residual variables
    u_minus = x[p + m : p + 2 * m]

    # calculate violations of u_plus >= 0
    u_plus_violation = np.maximum(-u_plus, 0.0)

    # calculate violations of u_minus >= 0
    u_minus_violation = np.maximum(-u_minus, 0.0)

    # calculate the average inequality error
    average_inequality_error = (
        np.sum(u_plus_violation) + np.sum(u_minus_violation)
    ) / m

    # calculate the original equality residual vector
    equality_error = A @ x - b

    # calculate the average equality error
    average_equality_error = np.mean(np.abs(equality_error))

    return (average_inequality_error, average_equality_error)

In [189]:
# solve the original LP using Gurobi automatic selection
start_time = time.perf_counter()

x_original, obj_original = solve_lp(
    A, b, c, lb, ub, method=-1, model_name="original_quantile_lp"
)

time_original_solve = time.perf_counter() - start_time

Set parameter Method to value -1
Set parameter OutputFlag to value 1
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 5000 rows, 10399 columns and 1606000 nonzeros (Min)
Model fingerprint: 0x37300496
Model has 10000 linear objective coefficients
Coefficient statistics:
  Matrix range     [2e-07, 1e+00]
  Objective range  [2e-01, 8e-01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-04, 1e+00]

Presolve time: 0.45s
Presolved: 5000 rows, 10399 columns, 1606000 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 0.05s

Barrier statistics:
 Dense cols : 399
 Free vars  : 399
 AA' NZ     : 1.596e+06
 Factor NZ  : 1.681e+06 (roughly 26 MB of memory)
 Factor Ops : 5.342e+08 (less than 1 second per iteration)
 Threads    : 8

                  Objective        

In [190]:
start_time = time.perf_counter()
x_apm, apm_iterations, apm_status = retrieve_apm(
    x0=x_tilde, A=A, b=b, lb=lb, ub=ub, max_iter=30, delta=0.01
)

time_apm_retrieval = time.perf_counter() - start_time

(
    apm_objective,
    apm_residual,
    apm_bound_violation,
) = evaluate_retrieval(
    x_apm,
    A,
    b,
    c,
    lb,
    ub,
)

print("APM objective:", apm_objective)
print("APM equality residual:", apm_residual)
print("APM bound violation:", apm_bound_violation)

APM objective: 20.39795507316918
APM equality residual: 350.7759493219652
APM bound violation: 1.386427654507335


In [191]:
if abs(obj_original) > 1e-12:
    relative_objective_error = abs(apm_objective - obj_original) / abs(
        obj_original
    )
    print("Relative objective error:", relative_objective_error)
else:
    relative_objective_error = np.nan
    print("Relative objective error: undefined because obj_original is 0")

Relative objective error: 0.981836797412805


In [192]:
def retrieve_dykstra(
    x0: np.ndarray,
    A,
    b: np.ndarray,
    lb: np.ndarray,
    ub: np.ndarray,
    max_iter: int = 30,
    delta: float = 0.01,
):
    """
    Apply the Dykstra APM logic from the authors' public code.
    """

    # first apply the authors' LSMR retrieval step
    x = retrieval_start(x0, A, b)

    # calculate one approximate solution of A x = b
    x_check = lsmr(A, b)[0]

    # initialize the two Dykstra correction vectors
    p = np.zeros_like(x)
    q = np.zeros_like(x)

    # initialize the bound-projected point
    y = np.zeros_like(x)

    # use the authors' default status until another condition is met
    status = "iteration limit"

    for iteration in range(1, max_iter + 1):
        # store the point at the beginning of the iteration
        x_previous = x.copy()

        # calculate the bound-error sum before the bound projection
        bound_error = bound_error_sum(x, lb, ub)

        # the authors stop immediately if there is no material bound error
        if bound_error <= delta:
            status = "no bound error"
            return x_previous, iteration, status

        # project x + p onto the bound set
        y = np.clip(x + p, lb, ub)

        # update the first Dykstra correction vector
        p = x + p - y

        # calculate the equality residual after the bound projection
        equality_error = float(np.linalg.norm(A @ y - b))

        # project y + q onto A x = b when needed
        if equality_error > delta:
            affine_input = y + q

            x = solve_projection_system(
                affine_input,
                A,
                b,
                x_check,
            )

            # update the second Dykstra correction vector
            q = y + q - x

        # calculate the authors' no-progress measure
        progress_solution = float(np.linalg.norm(x - x_previous))

        # stop when there is no sufficient progress
        if progress_solution <= delta:
            status = "no sufficient progress"
            return x, iteration, status

        # apply the authors' combined error stopping rule
        if bound_error + equality_error <= 2.0 * delta:
            status = "small total error"
            return x, iteration, status

    return x, max_iter, status

In [193]:
start_time = time.perf_counter()
x_dykstra, dykstra_iterations, dykstra_status = retrieve_dykstra(
    x0=x_tilde, A=A, b=b, lb=lb, ub=ub, max_iter=30, delta=0.01
)

time_dykstra_retrieval = time.perf_counter() - start_time

In [194]:
dykstra_objective, dykstra_residual, dykstra_bound_violation = (
    evaluate_retrieval(x_dykstra, A, b, c, lb, ub)
)

print("Dykstra objective:", dykstra_objective)
print("Dykstra equality residual:", dykstra_residual)
print("Dykstra bound violation:", dykstra_bound_violation)

Dykstra objective: 1246.6718448956117
Dykstra equality residual: 0.002109544615454236
Dykstra bound violation: 0.00034118451870401856


In [195]:
print("===== von Neumann APM =====")
print("Iterations:", apm_iterations)
print("Objective:", apm_objective)
print("Equality residual:", apm_residual)
print("Bound violation:", apm_bound_violation)

print("\n===== Dykstra APM =====")
print("Iterations:", dykstra_iterations)
print("Objective:", dykstra_objective)
print("Equality residual:", dykstra_residual)
print("Bound violation:", dykstra_bound_violation)

===== von Neumann APM =====
Iterations: 30
Objective: 20.39795507316918
Equality residual: 350.7759493219652
Bound violation: 1.386427654507335

===== Dykstra APM =====
Iterations: 12
Objective: 1246.6718448956117
Equality residual: 0.002109544615454236
Bound violation: 0.00034118451870401856


In [196]:
def retrieve_aarm(
    x0: np.ndarray,
    A,
    b: np.ndarray,
    lb: np.ndarray,
    ub: np.ndarray,
    max_iter: int = 30,
    delta: float = 0.01,
    alpha: float = 0.9,
    beta: float = 0.9,
):
    """
    Apply the AARM logic from the authors' public code.
    """

    # first apply the authors' LSMR retrieval step
    x = retrieval_start(x0, A, b)

    # calculate one approximate solution of A x = b
    x_check = lsmr(A, b)[0]

    # initialize the bound-projected point
    x_A = x.copy()

    # use the authors' default status until another condition is met
    status = "iteration limit"

    for iteration in range(1, max_iter + 1):
        # store the point at the beginning of the iteration
        x_previous = x.copy()

        # project onto the bound set
        x_A = np.clip(x, lb, ub)

        # calculate the first relaxed reflection
        # the authors' public code calculates this quantity
        # but does not use it in the subsequent update
        x_A_reflected = 2.0 * beta * x_A - x

        # project x_A onto A x = b
        x_B = solve_projection_system(
            x_A,
            A,
            b,
            x_check,
        )

        # calculate the second relaxed reflection
        x_B_reflected = 2.0 * beta * x_B - x

        # apply the averaged AARM operator
        x = (1.0 - alpha) * x + alpha * x_B_reflected

        # calculate the errors at the new point
        bound_error = bound_error_sum(
            x,
            lb,
            ub,
        )

        equality_error = float(np.linalg.norm(A @ x - b))

        # calculate the authors' no-progress measure
        progress_solution = float(np.linalg.norm(x - x_previous))

        # retain the public-code intermediate explicitly
        _ = x_A_reflected

        # stop when there is no sufficient progress
        if progress_solution <= delta:
            status = "no sufficient progress"
            return x, iteration, status

        # apply the authors' combined error stopping rule
        if bound_error + equality_error <= 2.0 * delta:
            status = "small total error"
            return x, iteration, status

    return x, max_iter, status

In [197]:
start_time = time.perf_counter()
x_aarm, aarm_iterations, aarm_status = retrieve_aarm(
    x0=x_tilde,
    A=A,
    b=b,
    lb=lb,
    ub=ub,
    max_iter=30,
    delta=0.01,
    alpha=0.9,
    beta=0.9,
)

time_aarm_retrieval = time.perf_counter() - start_time

print("AARM iterations:", aarm_iterations)
print("AARM stop reason:", aarm_status)

AARM iterations: 30
AARM stop reason: iteration limit


In [198]:
aarm_objective, aarm_residual, aarm_bound_violation = evaluate_retrieval(
    x_aarm, A, b, c, lb, ub
)

print("AARM objective:", aarm_objective)
print("AARM equality residual:", aarm_residual)
print("AARM bound violation:", aarm_bound_violation)

AARM objective: 873.1811835508081
AARM equality residual: 4.057419322305667
AARM bound violation: 0.1143687115390701


In [199]:
# Compare objectives only when ORIGINAL constraints are satisfied.
# The TLP objective is for a different, projected feasible set.
print("===== Original LP =====")
print("Objective:", obj_original)
print("\n===== TLP (projected LP) =====")
print("Objective:", obj_tilde)

for name, x, steps, status, objective, eq_residual, bound_error in [
    (
        "von Neumann APM",
        x_apm,
        apm_iterations,
        apm_status,
        apm_objective,
        apm_residual,
        apm_bound_violation,
    ),
    (
        "Dykstra APM",
        x_dykstra,
        dykstra_iterations,
        dykstra_status,
        dykstra_objective,
        dykstra_residual,
        dykstra_bound_violation,
    ),
    (
        "AARM",
        x_aarm,
        aarm_iterations,
        aarm_status,
        aarm_objective,
        aarm_residual,
        aarm_bound_violation,
    ),
]:
    feasible = retrieval_feasibility(x, A, b, lb, ub, tol=0.01)
    print(f"\n===== {name} =====")
    print("Iterations:", steps)
    print("Stop reason:", status)
    print("Original LP feasible at tolerance:", feasible)
    print(
        "Objective:",
        objective,
        "(comparable)" if feasible else "(infeasible point)",
    )
    print("Equality residual (L2):", eq_residual)
    print("Maximum bound violation:", bound_error)

# Wall-clock times for the same single problem instance.
# Shared data generation (D, b, A) is excluded from both pipelines.
print("\n===== Wall-clock time (seconds) =====")
print(f"Generate T:           {time_T_generation:.3f}")
print(f"Compute TA and Tb:    {time_projection:.3f}")
print(f"Solve TLP:            {time_TLP_solve:.3f}")
print(f"Solve original LP:    {time_original_solve:.3f}")
print(f"von Neumann retrieval:{time_apm_retrieval:.3f}")
print(f"Dykstra retrieval:    {time_dykstra_retrieval:.3f}")
print(f"AARM retrieval:       {time_aarm_retrieval:.3f}")

# Compare complete RP + retrieval workflows with the direct original solve.
time_projected_base = time_T_generation + time_projection + time_TLP_solve
print("\n===== Complete pipelines (seconds) =====")
print(f"Original LP directly:             {time_original_solve:.3f}")
print(
    f"RP + TLP + von Neumann retrieval: {time_projected_base + time_apm_retrieval:.3f}"
)
print(
    f"RP + TLP + Dykstra retrieval:     {time_projected_base + time_dykstra_retrieval:.3f}"
)
print(
    f"RP + TLP + AARM retrieval:        {time_projected_base + time_aarm_retrieval:.3f} (check feasibility above)"
)

# A ratio above 1 means the Dykstra pipeline ran faster than a direct LP solve.
dykstra_pipeline_time = time_projected_base + time_dykstra_retrieval
print(
    f"Direct / Dykstra pipeline:        {time_original_solve / dykstra_pipeline_time:.3f}x"
)

===== Original LP =====
Objective: 1123.0373594769978

===== TLP (projected LP) =====
Objective: 0.0

===== von Neumann APM =====
Iterations: 30
Stop reason: iteration limit
Original LP feasible at tolerance: False
Objective: 20.39795507316918 (infeasible point)
Equality residual (L2): 350.7759493219652
Maximum bound violation: 1.386427654507335

===== Dykstra APM =====
Iterations: 12
Stop reason: no sufficient progress
Original LP feasible at tolerance: False
Objective: 1246.6718448956117 (infeasible point)
Equality residual (L2): 0.002109544615454236
Maximum bound violation: 0.00034118451870401856

===== AARM =====
Iterations: 30
Stop reason: iteration limit
Original LP feasible at tolerance: False
Objective: 873.1811835508081 (infeasible point)
Equality residual (L2): 4.057419322305667
Maximum bound violation: 0.1143687115390701

===== Wall-clock time (seconds) =====
Generate T:           0.002
Compute TA and Tb:    0.016
Solve TLP:            0.089
Solve original LP:    2.353
von N

In [202]:
# get the dimensions of the quantile regression instance
m, p = D.shape

# calculate the Table 5 errors for von Neumann APM
apm_avgin, apm_avgeq = evaluate_errors(x_apm, A, b, m, p)

# calculate the Table 5 errors for Dykstra APM
dykstra_avgin, dykstra_avgeq = evaluate_errors(x_dykstra, A, b, m, p)

# calculate the Table 5 errors for AARM
aarm_avgin, aarm_avgeq = evaluate_errors(x_aarm, A, b, m, p)

# print the Table 5 feasibility measures
print("===== feasibility measures =====")

print("\nVon Neumann APM")
print("avgin:", apm_avgin)
print("avgeq:", apm_avgeq)

print("\nDykstra APM")
print("avgin:", dykstra_avgin)
print("avgeq:", dykstra_avgeq)

print("\nAARM")
print("avgin:", aarm_avgin)
print("avgeq:", aarm_avgeq)

===== feasibility measures =====

Von Neumann APM
avgin: 0.46979392877934567
avgeq: 3.9518830178263147

Dykstra APM
avgin: 0.00011154908347320331
avgeq: 2.379112005244452e-05

AARM
avgin: 0.038466912935840494
avgeq: 0.04965940782189908


In [ ]:
dykstra_bound_error_sum = bound_error_sum(
    x_dykstra,
    lb,
    ub,
)

dykstra_equality_error = np.linalg.norm(A @ x_dykstra - b)

dykstra_combined_error = dykstra_bound_error_sum + dykstra_equality_error

print(
    "Dykstra bound-error sum:",
    dykstra_bound_error_sum,
)

print(
    "Dykstra equality error:",
    dykstra_equality_error,
)

print(
    "Dykstra combined error:",
    dykstra_combined_error,
)

print(
    "Required combined error:",
    2.0 * 0.01,
)

Dykstra bound-error sum: 0.5577454173660166
Dykstra equality error: 0.002109544615454236
Dykstra combined error: 0.5598549619814708
Required combined error: 0.02


Dykstra stopped after 12 iterations because of insufficient progress.

The retrieved point is approximately feasible:
L2 equality residual = 0.00211,
average inequality violation = 0.0001115.

It is not exactly feasible because small bound violations remain.